<a href="https://colab.research.google.com/github/mythicalharshit/Deep-Learning/blob/main/EXP_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 20: Fine-Tune GPT-2 for a Specific Text Style and Generate Text

**Objective:** Load a pre-trained Large Language Model (GPT-2), fine-tune it on a custom text style dataset, and generate text based on user input.

**Language:** Python  
**Estimated Time:** 2 hours  
**Libraries Used:** transformers, datasets, torch, pandas


## 1. Introduction

GPT-2 is a pre-trained transformer-based language model. It can generate natural language text, and with fine-tuning, it can learn a custom writing style such as poetry, formal writing, technical writing, stories, quotes, or social media text.

In this experiment, we will:

1. Load a pre-trained GPT-2 model
2. Prepare a custom dataset containing a particular writing style
3. Tokenize the text
4. Fine-tune GPT-2 on that dataset
5. Save the model
6. Generate new text from user prompts


## 2. Install Required Libraries


In [18]:
# Run this once if packages are not installed
# !pip install transformers datasets accelerate torch pandas

## 3. Import Libraries


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file == "EXP_20.ipynb":
            print(os.path.join(root, file))

/content/drive/MyDrive/Colab Notebooks/EXP_20.ipynb


In [21]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/EXP_20.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove broken widget metadata
if "metadata" in nb and "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

print("Fixed notebook saved successfully.")

Fixed notebook saved successfully.


In [22]:
import os
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

Using device: cpu


## 4. Create or Load a Style Dataset

You can replace the sample texts below with your own dataset. For better fine-tuning, provide many examples in the style you want the model to learn.


In [23]:
# Example custom-style dataset
style_texts = [
    'The moon slipped quietly behind the clouds, and the village slept under a silver silence.',
    'In the old forest, every tree remembered a story the wind had once whispered.',
    'She wrote letters to the rain, hoping the sky would answer in thunder.',
    'The river carried secrets that only the night was patient enough to hear.',
    'A lantern glowed at the window like a small promise against the dark.',
    'Dreams arrived softly, wrapped in the scent of wet earth and jasmine.',
    'He walked through the dawn as if the light itself had been waiting for him.',
    'Even silence became music when the stars leaned close to listen.'
]

df = pd.DataFrame({'text': style_texts})
df.head()

,text
0,"The moon slipped quietly behind the clouds, an..."
1,"In the old forest, every tree remembered a sto..."
2,"She wrote letters to the rain, hoping the sky ..."
3,The river carried secrets that only the night ...
4,A lantern glowed at the window like a small pr...


## 5. Convert DataFrame to Hugging Face Dataset


In [24]:
dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['text'],
    num_rows: 8
})

## 6. Load GPT-2 Tokenizer and Model

We use `gpt2` as the base model. GPT-2 does not have a pad token by default, so we set the pad token equal to the end-of-sequence token.


In [25]:
model_name = 'gpt2'

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

print('Tokenizer and model loaded successfully.')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer and model loaded successfully.


## 7. Tokenize the Dataset


In [26]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=64,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_dataset

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 8
})

## 8. Data Collator

For causal language modeling, labels are created from the input tokens themselves.


In [27]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 9. Training Arguments

These settings keep the training small and simple for lab use. Increase epochs and dataset size for better results.


In [28]:
training_args = TrainingArguments(
    output_dir='./gpt2_finetuned_style',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=50,
    save_total_limit=1,
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

## 10. Fine-Tune GPT-2


In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.790076
20,2.688651


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=3.2393634796142576, metrics={'train_runtime': 113.8809, 'train_samples_per_second': 0.351, 'train_steps_per_second': 0.176, 'total_flos': 1306460160000.0, 'train_loss': 3.2393634796142576, 'epoch': 5.0})

## 11. Save the Fine-Tuned Model


In [30]:
save_path = './gpt2_finetuned_style_final'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print('Fine-tuned model saved at:', save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved at: ./gpt2_finetuned_style_final


## 12. Generate Text from User Input

This cell takes a prompt from the user and generates text in the learned style.


In [31]:
def generate_text(prompt, max_length=80):
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Example prompt
prompt = 'The evening sky'
generated_text = generate_text(prompt)
print('Prompt:', prompt)
print('\nGenerated Text:\n')
print(generated_text)

Prompt: The evening sky

Generated Text:

The evening sky hung in the distance as she and the other girls had watched the night. Even as they had been thinking, the silence continued. One word was heard above them all.

"You are not my girl, your sister."
, she whispered, her lips parting with the soft warmth of the earth. Herpian scent wafted over the quiet night as the words of


## 13. Interactive User Input

Run this cell and type your own prompt.


In [32]:
user_prompt = input('Enter your prompt: ')
print('\nGenerated Output:\n')
print(generate_text(user_prompt))

Enter your prompt: heyy

Generated Output:

heyy had fallen on the edge of a dream and he wanted to hear it. His eyes were dark and cold, but his thoughts were warm. He remembered the day he remembered when he had his hand wrapped around the small boy's shoulders.

"Let's go to sleep," he said. They drifted out through the silence that followed, and his heart beat as if he knew how to


## 14. Notes for Better Fine-Tuning

- Use a much larger dataset for better style learning.
- Clean and format the text consistently.
- Increase training epochs if loss is still high.
- You can replace the sample texts with poetry, technical paragraphs, dialogues, news, or any other style.
- Fine-tuning large models may require a GPU for faster training.


## 15. Conclusion

In this experiment, we loaded the pre-trained GPT-2 model, fine-tuned it on a small custom-style dataset, and generated text from user prompts. This demonstrates how a general language model can be adapted to produce content in a more specific writing style.
